In [1]:
import os
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
DB_URL = os.getenv("LOCAL_DATABASE_URL")
if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DB_URL)
pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

print("Database connection berhasil")
print("Pipeline run ID:", pipeline_run_id)


Database connection berhasil
Pipeline run ID: 20260924075413


## 1. Build `channel_campaign_daily`

In [3]:
order_360 = pd.read_sql(
    "SELECT * FROM gold.order_360",
    engine
)

order_360.to_sql(
    "_tmp_order_360",
    engine,
    if_exists="replace",
    index=False
)

channel_campaign_daily = pd.read_sql("""
WITH web AS (
    SELECT DISTINCT
        event_id,
        "payload.order_id" AS order_id,
        "payload.customer_id" AS customer_id,
        "payload.campaign_id" AS campaign_id,
        "payload.channel" AS sales_channel,
        occurred_at_utc
    FROM silver.web_events
    WHERE "payload.campaign_id" IS NOT NULL
),

attr_ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY occurred_at_utc DESC, event_id DESC
        ) AS rn
    FROM web
    WHERE order_id IS NOT NULL
),

attr AS (
    SELECT
        o.order_date::date AS metric_date,
        a.sales_channel,
        a.campaign_id,
        a.order_id,
        o.customer_id,
        o.gross_merchandise_value AS gross_revenue,
        o.net_revenue,
        o.refunded_amount AS refunds
    FROM attr_ranked a
    JOIN _tmp_order_360 o
        ON o.order_id = a.order_id
    WHERE a.rn = 1
),

agg AS (
    SELECT
        metric_date,
        sales_channel,
        campaign_id,
        COUNT(DISTINCT order_id) AS attributed_orders,
        COUNT(DISTINCT customer_id) AS attributed_customers,
        SUM(gross_revenue) AS gross_revenue,
        SUM(net_revenue) AS net_revenue,
        SUM(refunds) AS refunds
    FROM attr
    GROUP BY
        metric_date,
        sales_channel,
        campaign_id
),

spend AS (
    SELECT
        spend_date::date AS metric_date,
        channel AS sales_channel,
        campaign_id,
        SUM(spend_amount) AS campaign_spend
    FROM silver.campaign_spend
    GROUP BY
        spend_date::date,
        channel,
        campaign_id
),

sessions AS (
    SELECT
        occurred_at_utc::date AS metric_date,
        "payload.channel" AS sales_channel,
        "payload.campaign_id" AS campaign_id,
        COUNT(DISTINCT "payload.session_id") AS sessions
    FROM silver.web_events
    WHERE "payload.campaign_id" IS NOT NULL
    GROUP BY
        occurred_at_utc::date,
        "payload.channel",
        "payload.campaign_id"
),

grid AS (
    SELECT
        metric_date,
        sales_channel,
        campaign_id
    FROM spend

    UNION

    SELECT
        metric_date,
        sales_channel,
        campaign_id
    FROM agg
)

SELECT
    g.metric_date,
    g.sales_channel,
    g.campaign_id,

    COALESCE(
        s.campaign_spend,
        0
    ) AS campaign_spend,

    COALESCE(
        a.attributed_orders,
        0
    ) AS attributed_orders,

    COALESCE(
        a.attributed_customers,
        0
    ) AS attributed_customers,

    COALESCE(
        a.gross_revenue,
        0
    ) AS gross_revenue,

    COALESCE(
        a.net_revenue,
        0
    ) AS net_revenue,

    COALESCE(
        a.refunds,
        0
    ) AS refunds,

    CASE
        WHEN COALESCE(s.campaign_spend, 0) = 0
            THEN NULL
        ELSE
            COALESCE(a.net_revenue, 0)
            / s.campaign_spend
    END AS roas,

    CASE
        WHEN COALESCE(se.sessions, 0) = 0
            THEN NULL
        ELSE
            COALESCE(a.attributed_orders, 0)::numeric
            / se.sessions
    END AS conversion_rate

FROM grid g

LEFT JOIN spend s
    USING (
        metric_date,
        sales_channel,
        campaign_id
    )

LEFT JOIN agg a
    USING (
        metric_date,
        sales_channel,
        campaign_id
    )

LEFT JOIN sessions se
    USING (
        metric_date,
        sales_channel,
        campaign_id
    )

ORDER BY
    g.metric_date,
    g.sales_channel,
    g.campaign_id

""", engine)

# Ensure integer contract columns
for c in [
    "attributed_orders",
    "attributed_customers"
]:
    channel_campaign_daily[c] = (
        channel_campaign_daily[c]
        .fillna(0)
        .astype(int)
    )

print(
    "channel_campaign_daily:",
    len(channel_campaign_daily),
    "rows"
)

channel_campaign_daily: 2586 rows


## 2. Contract types + pipeline_run_id

In [4]:
channel_campaign_daily["pipeline_run_id"] = pipeline_run_id


## 3. Recreate target table sesuai `gold.sql`

In [5]:
gold_ddl = """CREATE SCHEMA IF NOT EXISTS gold;
DROP TABLE IF EXISTS gold.channel_campaign_daily;
CREATE TABLE gold.channel_campaign_daily (
    metric_date DATE NOT NULL,
    sales_channel TEXT NOT NULL,
    campaign_id TEXT NOT NULL,
    campaign_spend NUMERIC(14,2) NOT NULL,
    attributed_orders INTEGER NOT NULL,
    attributed_customers INTEGER NOT NULL,
    gross_revenue NUMERIC(14,2) NOT NULL,
    net_revenue NUMERIC(14,2) NOT NULL,
    refunds NUMERIC(14,2) NOT NULL,
    roas NUMERIC(14,4),
    conversion_rate NUMERIC(14,4),
    pipeline_run_id TEXT NOT NULL,
    PRIMARY KEY (metric_date, sales_channel, campaign_id)
);"""
with engine.begin() as conn:
    conn.execute(text(gold_ddl))

channel_campaign_daily.to_sql("channel_campaign_daily", engine, schema="gold", if_exists="append", index=False, method="multi")
print("gold.channel_campaign_daily berhasil dibuat.")

gold.channel_campaign_daily berhasil dibuat.


## 4. Final validation

In [6]:
df = pd.read_sql("SELECT * FROM gold.channel_campaign_daily", engine)
print("Rows:", len(df))
print("Unique date-channel-campaign:", df[["metric_date","sales_channel","campaign_id"]].drop_duplicates().shape[0])
print("Duplicate rows:", len(df) - len(df.drop_duplicates()))
print("NULL values:", int(df.isna().sum().sum()))


Rows: 2586
Unique date-channel-campaign: 2586
Duplicate rows: 0
NULL values: 1766
